# 黑马头条项目｜第05天：延迟队列精确发布文章

> 学习目标：理解延迟任务，把文章提交后的直接审核改为定时审核；建立 MySQL 持久化、Redis 缓存、调度服务、Feign 消费和文章审核之间的完整链路。

本篇依据百度网盘 `day05-延迟队列精确发布文章/讲义/延迟任务精准发布文章.md` 的网页预览整理，延续第03天文章发布和第04天自动审核。未读取配套源码文件，未启动课程服务，也未执行数据库、Redis或第三方审核接口测试。下文代码为讲义关键逻辑的学习摘录或明确标注的修正示意，不是可以直接运行的完整工程。

**先明确“精准发布”的含义：讲义让任务到时后才开始审核，再保存 APP 文章。** 它采用每分钟搬迁到期任务、每秒拉取任务的方式，发布时间还会受到审核耗时、积压和服务延迟影响，不能理解成文章必定在指定毫秒上线。

资料：[黑马头条项目分享链接](https://pan.baidu.com/s/1u7PNExeoEsYGh5pAnSyocg?pwd=9987)。


## 1. 业务问题：为什么需要延迟任务

第04天的路径是“提交文章 → 异步审核 → 保存 APP 文章”。如果用户设置了未来的发布时间，就需要把审核入口延后。

```text
第04天：提交 → 立即发起审核 → 审核通过 → 创建/更新 APP 文章
第05天：提交 → 登记任务 → 等到 publishTime → 发起审核 → 创建/更新 APP 文章
```

定时任务通常按固定周期或指定时刻触发，例如每分钟扫描一次；延迟任务由某个业务事件产生，在事件之后的目标时间执行，例如下单30分钟未支付就取消订单。两者可以组合：本课用定时扫描驱动延迟队列。

| 场景 | 任务产生 | 到期动作 | 提前终止条件 |
|---|---|---|---|
| 文章定时发布 | 提交并设置发布时间 | 拉起文章审核 | 撤稿、改期或取消提交 |
| 超时订单关闭 | 创建订单 | 检查未付款并关闭 | 支付成功后取消关闭任务 |
| 接口失败重试 | 调用失败 | 再次调用 | 已成功或超过最大次数 |

延迟队列解决的是“何时让任务可以被执行”，业务处理还需要回答“是否仍该执行、是否执行成功、失败如何重试”。例如订单到期时仍需检查最新支付状态，而不是收到任务就无条件关单。


## 2. 技术选择：DelayQueue、RabbitMQ、Redis

| 方案 | 基本机制 | 适用特点 | 需要注意 |
|---|---|---|---|
| JDK `DelayQueue` | `PriorityQueue` 排序，元素实现 `Delayed`，到期才能取出 | 单进程内调度，使用简单 | 任务在内存中，进程重启丢失；跨实例共享需要另外设计 |
| RabbitMQ | 消息 TTL 到期，经死信交换机进入消费队列 | 能复用消息中间件和消费机制 | TTL、死信路由与确认要配合；不同到期时间可能遇到队头阻塞 |
| Redis | ZSet 按到期时间排序，List 保存可消费任务 | 容易按任务类型分组，可结合数据库恢复 | 持久化、原子搬迁、消费确认和重试都需要设计 |

讲义最终选择 **MySQL + Redis**：数据库保存任务，Redis管理近期调度。RabbitMQ在本日是方案比较，并没有实现完整的消息投递流程。

### DelayQueue 的时间单位修正

讲义示例的 `getDelay(TimeUnit unit)` 没有按 `unit` 转换，并用 `int` 保存秒级时间，容易把秒和纳秒混用。下面是遵守接口时间单位的学习示意：

```java
final class DelayedTask implements Delayed {
    private final long deadlineNanos;

    DelayedTask(long delayMillis) {
        this.deadlineNanos = System.nanoTime()
                + TimeUnit.MILLISECONDS.toNanos(delayMillis);
    }

    @Override
    public long getDelay(TimeUnit unit) {
        return unit.convert(deadlineNanos - System.nanoTime(),
                            TimeUnit.NANOSECONDS);
    }

    @Override
    public int compareTo(Delayed other) {
        return Long.compare(deadlineNanos,
                            ((DelayedTask) other).deadlineNanos);
    }
}
```

该示意假设队列内都是 `DelayedTask`。`System.nanoTime()` 适合进程内计算剩余时长，不能拿它做跨进程持久化时间；课程 DTO 的 `executeTime` 则使用 Unix 毫秒时间戳。


## 3. Redis 延迟队列的整体结构

```text
                      新任务
                         │
                         ▼
                MySQL taskinfo + 日志
                         │
            ┌────────────┼─────────────────┐
            │            │                 │
        已到执行时间    未来5分钟以内       5分钟以外
            │            │                 │
            ▼            ▼                 ▼
      Redis List     Redis ZSet         暂存在 MySQL
      topic_<分组>   future_<分组>           │
            ▲            │                 │
            │     每分钟扫描到期任务       │
            └────────────┘      每5分钟 reloadData 纳入缓存
            │
        自媒体每秒 poll
            │
        审核 → APP文章保存 → 回写自媒体文章
```

缓存分组由 `taskType + "_" + priority` 组成，例如文章审核任务：

| 位置 | 示例 | 保存内容 |
|---|---|---|
| 数据库 | `taskinfo` | 所有仍待调度的任务，包含远期任务 |
| 延迟集合 | `future_1001_1` | 未来5分钟内的任务，score 为 `executeTime` |
| 就绪队列 | `topic_1001_1` | 已到期任务的序列化数据 |
| 数据库日志 | `taskinfo_logs` | 历史任务、状态和乐观锁版本 |

List 采用 `LPUSH` 入队、`RPOP` 出队，在单队列的正常入队顺序下形成 FIFO。ZSet按score排序；时间相同的任务并不天然保证业务提交顺序。

`priority` 在本课实现中是队列分组的一部分，消费者指定类型和优先级拉取。**不能由此推断不同队列之间存在“高优先级总先执行”的全局调度规则。**


## 4. 建立通用调度微服务

新增模块 `heima-leadnews-schedule`，服务注册名为 `leadnews-schedule`，使用数据库 `leadnews_schedule`。它提供通用任务添加、取消、拉取接口；文章审核属于自媒体服务的业务，调度服务不直接处理文章内容。

核心依赖包括课程公共模块、MyBatis-Plus、Redis，以及 Redis 连接池使用的 `commons-pool2`。公共 `CacheService` 封装 List、ZSet、SCAN、Pipeline和锁相关操作，通过公共模块自动装配供调度服务使用。

配置结构示意，实际属性应与课程使用的 Spring Boot 版本和配置中心一致：

```yaml
spring:
  application:
    name: leadnews-schedule
  datasource:
    url: ${SCHEDULE_DB_URL}
    username: ${SCHEDULE_DB_USERNAME}
    password: ${SCHEDULE_DB_PASSWORD}
  redis:
    host: ${REDIS_HOST}
    port: ${REDIS_PORT}
    password: ${REDIS_PASSWORD}
```

调度服务启用 `@EnableScheduling`，否则缓存刷新和定期恢复方法不会自动执行。自媒体服务也需要启用调度来执行消费轮询，并保留异步任务所需的 `@EnableAsync`。


## 5. 持久化模型、任务 DTO 与状态

### 5.1 两张表承担不同责任

| 字段 | `taskinfo` | `taskinfo_logs` | 含义 |
|---|---|---|---|
| `taskId` | 有 | 有 | Long 任务ID，课程采用雪花算法 |
| `executeTime` | 有 | 有 | 数据库日期时间 |
| `parameters` | 有 | 有 | 业务参数字节数组 |
| `priority` | 有 | 有 | 优先级/队列分组参数 |
| `taskType` | 有 | 有 | 任务类型 |
| `status` | 无 | 有 | 任务状态 |
| `version` | 无 | 有 | 乐观锁版本 |

讲义状态常量：`SCHEDULED = 0`（已安排）、`EXECUTED = 1`（已执行）、`CANCELLED = 2`（已取消）。处理结束或取消时删除待调度表记录，日志表继续保留历史。

```java
// 课程使用的 MyBatis-Plus 写法，版本升级时核对 ID 策略
@TableId(value = "task_id", type = IdType.ID_WORKER)
private Long taskId;

// 日志实体
@Version
private Integer version;
```

`OptimisticLockerInnerInterceptor` 配合 `@Version` 检查并发更新，但仍须检查更新影响行数；它不能自动保证 Redis 入队、任务领取和业务执行不重复。

### 5.2 对外 DTO

```java
@Data
public class Task {
    private Long taskId;
    private Integer taskType;
    private Integer priority;
    private long executeTime;   // 毫秒时间戳，不是执行ID
    private byte[] parameters;  // 业务参数，调度服务不解释其内容
}

public interface TaskService {
    long addTask(Task task);
    boolean cancelTask(long taskId);
    Task poll(int type, int priority);
}
```

数据库 `Date` 与 DTO 毫秒时间戳互相转换时要保持单位一致。业务参数只携带执行所需的信息，例如文章ID；不要把密钥或完整文章正文复制进通用任务。


## 6. 添加任务：先落库，再进入对应缓存

`addTask` 的讲义流程：

1. 保存 `taskinfo`，生成 `taskId`；
2. 保存 `taskinfo_logs`，初始化 `status = 0`、`version = 1`；
3. 将任务ID回填 DTO；
4. 根据执行时间决定是否写 Redis；
5. 返回任务ID。

下面是缓存分流逻辑的整理示意，实际调用以 `CacheService` 为准：

```java
private void addTaskToCache(Task task) {
    long now = System.currentTimeMillis();
    long horizon = now + TimeUnit.MINUTES.toMillis(5);
    String group = task.getTaskType() + "_" + task.getPriority();
    String payload = JSON.toJSONString(task);

    if (task.getExecuteTime() <= now) {
        cacheService.lLeftPush("topic_" + group, payload);
    } else if (task.getExecuteTime() <= horizon) {
        cacheService.zAdd("future_" + group, payload,
                          task.getExecuteTime());
    }
    // 更远的任务已经保存在 MySQL，等待后续恢复/加载。
}
```

| 到期时间 | 立即进入哪里 | 理由 |
|---|---|---|
| 已到期或立即执行 | List | 已具备消费资格 |
| 当前时间之后5分钟内 | ZSet | 等待时间到达 |
| 5分钟之后 | 仅数据库 | 缩小 Redis 缓存窗口 |

**事务边界：** 两张 MySQL 表应在同一数据库事务中完成。Redis写入不属于该数据库事务，因此可能出现“数据库成功、Redis失败”。本课用后续加载恢复，生产可用事务消息表/Outbox配合补偿。

讲义部分逻辑捕获异常后不再抛出，可能使外层事务正常提交，留下不完整数据。数据库失败应回滚并向调用方报告；数据库已经成功但缓存失败时，应留下可重试的记录，而不是声称所有步骤均成功。


## 7. 取消任务：修正讲义中的状态和返回值

取消应删除待调度记录、更新日志为取消状态，并清理缓存。讲义示例有两处直接影响行为的问题：

| 讲义写法 | 问题 | 正确意图 |
|---|---|---|
| `updateDb(taskId, EXECUTED)` | 把取消记成执行 | 使用 `CANCELLED` |
| 最后始终 `return false` | 即使取消成功也返回失败 | 返回实际结果 |

对应的**最小修正示意**如下，仍沿用讲义的数据库和缓存辅助方法：

```java
@Override
public boolean cancelTask(long taskId) {
    Task task = updateDb(taskId, ScheduleConstants.CANCELLED);
    if (task == null) {
        return false;
    }
    removeTaskFromCache(task);
    return true;
}
```

讲义的 `updateDb` 删除 `taskinfo`，更新日志状态，再从日志信息还原 `Task`。真实实现要处理不存在、已执行、已取消、并发更新失败，并明确重复取消是否返回成功。

缓存删除如果仅根据“现在是否已到期”选择 List或ZSet，可能与搬迁任务发生竞争：判断时在ZSet，删除时已经进入List。因此可以同时清理两端，并在消费前复核数据库取消状态；需要更严格的一致性时使用原子操作和状态迁移。

即便成功删除缓存，也无法撤回已经交给消费者执行的任务。文章撤稿/改期还必须在业务执行入口检查最新状态，并通过任务ID或版本区分新旧任务。


## 8. 拉取任务：RPOP 之后不等于业务执行成功

课程 `poll(type, priority)` 的主线：

```text
拼接 topic_<type>_<priority>
           ↓
RPOP（没有数据就返回空）
           ↓
JSON还原 Task
           ↓
删除 taskinfo，日志改为 EXECUTED
           ↓
返回给调用方，调用方开始业务处理
```

学习示意：

```java
public Task poll(int type, int priority) {
    String key = "topic_" + type + "_" + priority;
    String payload = cacheService.lRightPop(key);
    if (StringUtils.isBlank(payload)) {
        return null;
    }
    Task task = JSON.parseObject(payload, Task.class);
    updateDb(task.getTaskId(), ScheduleConstants.EXECUTED);
    return task;
}
```

`RPOP` 单个操作是原子的，同一个List元素不会被两个消费者同时弹出；但重复入队的同一业务任务仍可能被重复执行。

**本课的 EXECUTED 实际更接近“已经交付/取走”。** 自媒体收到任务之前，调度服务已删除待任务记录。若响应丢失、消费端崩溃、反序列化失败或审核异常，这个任务不会因为 `taskinfo` 恢复而自动回来。

生产改进应增加明确的领取与确认协议，例如：

```text
待执行 → 已领取/执行中 → 业务成功 → ACK → 已完成
                   └→ 超时或失败 → 重试/人工处理
```

处理中列表、Redis Streams消费组或带租约的数据库任务领取都可以承载这种协议；无论选择哪种，业务处理需要幂等，并在超过重试上限后留下可追踪的失败记录。


## 9. 刷新到期任务：SCAN、ZSet 与 Pipeline

调度服务每分钟扫描 `future_*`，选出 `executeTime <= now` 的任务，移入对应 `topic_*`：

```java
@Scheduled(cron = "0 */1 * * * ?")
public void refresh() {
    // 分布式部署时先获取刷新锁。
    // SCAN future_*，对各分组读取 score <= now 的任务。
    // 从 ZSet 移除，再 LPUSH 到同组 List。
}
```

### 9.1 为什么用 SCAN

`KEYS future_*` 一次遍历所有匹配键，数据量大时会长时间占用 Redis。`SCAN` 使用游标分批遍历，单次工作量较可控，但它不是固定快照，遍历期间可能返回重复键；调用方仍要做好去重和并发处理。

### 9.2 为什么用 Pipeline

逐个发送命令需要多次网络往返；Pipeline批量发送命令能减少往返和等待。讲义的 `refreshWithPipeline` 把 `ZREM` 和 `LPUSH` 批处理，以提高效率。

**Pipeline不提供事务原子性。** `ZREM`成功后、`LPUSH`执行前崩溃，会暂时丢失缓存任务；多实例先读取相同成员再搬迁，也可能重复入队。不能把“批量发送”解释为“全部成功或全部失败”。

### 9.3 原子搬迁的扩展示意

下面Lua脚本是笔记补充，不是讲义原实现。它在同一脚本中分批读取、删除、入队，删除成功才入队：

```lua
-- KEYS[1]: future队列；KEYS[2]: topic队列
-- ARGV[1]: 当前毫秒时间；ARGV[2]: 单批最大任务数
local items = redis.call('ZRANGEBYSCORE', KEYS[1], '-inf', ARGV[1],
                         'LIMIT', 0, tonumber(ARGV[2]))
local moved = 0
for _, item in ipairs(items) do
    if redis.call('ZREM', KEYS[1], item) == 1 then
        redis.call('LPUSH', KEYS[2], item)
        moved = moved + 1
    end
end
return moved
```

批量上限用于避免长脚本阻塞。Redis Cluster下两个key还必须落在同一slot，例如使用共同的 `{1001_1}` hash tag。原子搬迁解决的是Redis内部这一段的竞争，不能覆盖数据库更新或远程审核。


## 10. 多实例刷新与分布式锁

单实例只有一个刷新入口；多实例部署后，每台调度服务都会触发同一个定时方法。因此讲义让实例先竞争 `FUTURE_TASK_SYNC` 对应的锁，拿到锁的实例才刷新缓存。

锁的核心是一个原子的 `SET key token NX PX ttl`：

```text
key   = FUTURE_TASK_SYNC_lock
token = 每次加锁生成的唯一 UUID
NX    = 仅当 key 不存在时设置
PX    = 以毫秒设置过期时间，讲义使用30秒
```

`CacheService.tryLock(name, expire)` 为name补 `_lock` 后缀，生成token，利用 Redis `SET_IF_ABSENT` 与毫秒过期时间加锁；成功返回token，失败返回空。

**不要先 SETNX、再单独 EXPIRE。** 两条命令之间进程退出可能遗留永久锁。讲义使用同一条SET表达加锁与过期，更合适。

讲义依赖锁自动过期，没有显式解锁。需要主动释放时，只能释放自己持有的锁；下面为补充的原子校验释放示意：

```lua
if redis.call('GET', KEYS[1]) == ARGV[1] then
    return redis.call('DEL', KEYS[1])
end
return 0
```

| 风险 | 为什么发生 | 处理思路 |
|---|---|---|
| 工作超过30秒 | 锁过期后另一实例进入 | 限制批次、合理TTL、续期或带租约的成熟实现 |
| 删除他人的锁 | 旧持有者恢复后直接DEL | 用token校验并原子删除 |
| 锁持有者失联 | 不能主动释放 | 过期时间保障后续可竞争 |
| 任务仍重复 | 锁只保护一段调度流程 | 业务幂等和状态校验 |

锁能减少并发竞争，不能直接承诺“任务只执行一次”。刷新、重新加载与取消若没有协调，仍存在交叉影响。


## 11. 缓存恢复：启动加载与每5分钟 reloadData

讲义在启动时使用 `@PostConstruct` 加载，并通过每5分钟的调度再次加载：

```java
@PostConstruct
@Scheduled(cron = "0 */5 * * * ?")
public void reloadData() {
    // 1. 清除 future_* 与 topic_* 缓存。
    // 2. 查询 executeTime < 当前时间 + 5分钟 的 taskinfo。
    // 3. 把查询结果转换为 Task，调用 addTaskToCache。
}
```

它承担两个作用：把原来超过5分钟的远期任务纳入缓存窗口；Redis重启或缓存写入失败后，根据数据库待任务恢复缓存。历史日志已标执行且待任务被删除的任务不会通过该查询恢复。

### 全量清缓存的局限

讲义先清队列再加载，易于演示，但线上会与新增、搬迁和消费竞争：

1. 新任务刚写Redis，就被另一实例的清理删除；
2. 任务已弹出但数据库状态尚未更新，又被加载入队；
3. 清理后数据库查询失败，出现队列空窗；
4. 多实例同时启动或定期加载，互相覆盖缓存；
5. 数据较多时，一次查询和全量写入造成压力。

生产可改为按ID去重的增量对账、分批扫描及缓存补偿，结合分布式锁协调加载，并避免直接清除正在消费的共享队列。ZSet去重以成员字符串为依据，并非天然按 `taskId` 去重；JSON内容变化后可能是另一个成员，List也没有去重能力。

本方案5分钟窗口与5分钟加载频率在边界附近会受到执行延迟影响，远期任务可能较晚进入缓存。数据库时间索引、窗口适当重叠、幂等入队与监控能降低漏扫/迟到风险。


## 12. 通过 Feign 暴露通用调度接口

公共API模块定义 `IScheduleClient`，调度服务控制器实现对应接口，并以 `ResponseResult` 包装结果。

```java
@FeignClient("leadnews-schedule")
public interface IScheduleClient {
    @PostMapping("/api/v1/task/add")
    ResponseResult addTask(@RequestBody Task task);

    @GetMapping("/api/v1/task/cancel/{taskId}")
    ResponseResult cancelTask(@PathVariable("taskId") long taskId);

    @GetMapping("/api/v1/task/poll/{type}/{priority}")
    ResponseResult poll(@PathVariable("type") int type,
                        @PathVariable("priority") int priority);
}
```

以上是讲义的接口路径和HTTP方法。取消会修改状态，poll会弹出元素，两者都有副作用；生产接口可将取消改为POST/DELETE、领取改为POST，避免GET语义造成缓存、预取或错误重试。

| 接口 | 成功结果 | 调用方要处理 |
|---|---|---|
| add | 新任务ID | 保存ID；超时后查询/幂等重试，避免重复创建 |
| cancel | 是否取消成功 | 已执行/已取消/不存在的语义 |
| poll | Task或空结果 | 空队列、错误响应、响应丢失、参数异常 |

Feign序列化返回的 `data` 不一定已是Java `Task`，需要按课程ResponseResult结构转换，检查状态码和空值。应设置超时，并区分“服务失败”与“队列暂时没有任务”。


## 13. 文章任务类型与 Protostuff 参数

讲义通过枚举统一任务类型和优先级：

```java
NEWS_SCAN_TIME(1001, 1, "文章定时审核"),
REMOTEERROR(1002, 2, "第三方接口调用失败，重试");
```

文章任务无需保存完整文章，讲义构造仅含id的 `WmNews`，将其编码到 `Task.parameters`。真正审核时再查数据库，取得最新标题、内容和状态。

```text
WmNews（仅id）
     ↓ ProtostuffUtil.serialize
byte[] parameters
     ↓ 保存/传输 Task
byte[] parameters
     ↓ ProtostuffUtil.deserialize
WmNews（取得id）→ 查数据库 → 执行审核
```

讲义使用 `protostuff-core` 和 `protostuff-runtime`，版本为 `1.6.0`。Protostuff是二进制序列化工具；与JDK对象序列化相比，选型时应实际比较报文大小、性能、兼容性和维护成本，不应只凭工具名称判断优劣。

```java
task.setParameters(ProtostuffUtil.serialize(wmNews));

WmNews wmNews = ProtostuffUtil.deserialize(
        task.getParameters(), WmNews.class);
```

以上调用依赖课程工具类。生产参数可使用独立DTO，例如 `newsId + publishVersion + schemaVersion`；修改实体字段时注意旧任务能否反序列化。接口接收的任务类型、参数长度和时间应校验，禁止任意调用方借任务服务执行不受控业务。


## 14. 自媒体提交：把直接审核改成登记任务

新增 `WmNewsTaskService`：

```java
public interface WmNewsTaskService {
    void addNewsToTask(Integer id, Date publishTime);
    void scanNewsByTask();
}
```

讲义的登记逻辑如下，枚举取值和序列化使用课程工具：

```java
@Async
@Override
public void addNewsToTask(Integer id, Date publishTime) {
    Task task = new Task();
    task.setExecuteTime(publishTime.getTime());
    task.setTaskType(TaskTypeEnum.NEWS_SCAN_TIME.getTaskType());
    task.setPriority(TaskTypeEnum.NEWS_SCAN_TIME.getPriority());

    WmNews parameters = new WmNews();
    parameters.setId(id);
    task.setParameters(ProtostuffUtil.serialize(parameters));
    scheduleClient.addTask(task);
}
```

提交服务保留第03天的文章保存、草稿跳过、内容和封面素材关系处理，将第04天直接调用审核的入口改为：

```java
// 原入口：wmNewsAutoScanService.autoScanWmNews(wmNews.getId());
wmNewsTaskService.addNewsToTask(wmNews.getId(), wmNews.getPublishTime());
```

### 落地时必须补齐的边界

- 草稿不添加定时审核任务；提交后的状态应能被第04天审核入口识别。
- `publishTime` 为空时要按业务约定设为立即执行或返回校验错误，不能直接触发空指针。
- 先提交文章数据库事务，再登记任务；`@Async`自身并不保证“事务提交后才执行”。可使用提交后事件或事务消息表。
- 检查add的响应并保存taskId，必要时使用独立关联表，便于取消和改期。讲义示例未检查响应，不能据此认为任务一定登记成功。
- 异步失败要有日志、重试与监控；Feign超时可能已在服务端添加成功，重试应使用业务幂等键。

`@Async`通常依赖代理调用；同类内部直接调用注解方法不一定进入异步线程。生产线程池还需要有界队列、拒绝策略与失败处理。


## 15. 每秒消费文章任务，并接回第04天审核

自媒体服务开启 `@EnableScheduling`，每秒拉取文章任务分组：

```java
@Scheduled(fixedRate = 1000)
@Override
public void scanNewsByTask() {
    // 1. scheduleClient.poll(1001, 1)
    // 2. 校验 ResponseResult 状态码与 data 非空
    // 3. 将 data 转为 Task
    // 4. Protostuff反序列化，取得 WmNews.id
    // 5. wmNewsAutoScanService.autoScanWmNews(id)
}
```

`fixedRate = 1000` 表示以一秒为周期尝试调度；实际开始时间还受到线程池和处理耗时影响。讲义每次只取一个任务，负载较大时会积压，需要结合审核线程池容量进行限量批量领取，而不是无限拉取。

完整业务链路：

```text
文章入库 + 素材关联
        ↓ 提交后登记
任务落库 → ZSet等到期 → List就绪 → Feign poll
        ↓
取出文章ID → 查询最新 wm_news
        ↓
敏感词 + 文本内容安全 + 图片/OCR审核
        ├─ 失败：状态2
        ├─ 转人工：状态3
        └─ 通过：调用article服务保存文章
                    ↓
              回填article_id，发布状态9
                    ↓
              按第04天流程生成静态页
```

第04天的状态 `8` 表示审核通过、待发布，但**本日登记的是“到点开始审核”任务**，不能将其写成“先审核变为8，到点直接发布”。如果业务要求提前审核并在指定时间上线，需要另外设计审核与发布分离的流程。

任务到期只是执行的必要条件；审核入口还应复核文章仍存在、未撤稿、处于可审核状态、发布时间未被修改、任务版本仍有效。APP保存及回填要有幂等控制，避免重复领取或重试创建重复文章。


## 16. 时间精度、改期与一致性

### 16.1 发布时间为什么可能偏晚

```text
最终上线延迟
  ≈ 缓存加载延迟 + 到期扫描等待 + 拉取等待
    + 队列积压 + 审核/跨服务保存耗时
```

通常已在ZSet中的任务，可能等待接近一分钟才被refresh搬迁，再等待消费者拉取；每秒轮询无法消除前面的分钟级等待。Redis故障、重载空窗或待任务积压时可能更长，因此应监控实际延迟，而不能承诺固定上限。

可以提高刷新频率、按最近到期时间唤醒或提前审核，但需要同时考虑CPU、Redis压力和业务状态机。各实例时钟也应同步，并统一 `executeTime` 的毫秒单位。

### 16.2 修改发布时间应处理旧任务

```text
保存文章新版本/新时间
        ↓
取消旧任务 + 登记新任务（保存关联ID）
        ↓
消费时按文章ID和发布版本复核
```

取消旧任务和创建新任务跨服务，不能视为天然原子操作。即使旧任务已被取出，新版本复核也应让其跳过；创建新任务失败需补偿。只更新 `wm_news.publish_time` 而保留旧任务，会导致按旧时间启动审核。

### 16.3 哪些成功不能混为一谈

| 事件 | 只证明什么 |
|---|---|
| 文章提交成功 | 文章数据库保存完成 |
| 任务添加成功 | 调度端接受了任务 |
| 任务出队 | 消费者可能获得了任务 |
| 审核通过 | 内容满足审核规则 |
| APP文章保存成功 | 文章端数据落库 |
| 静态页生成成功 | 详情页产物已准备好 |

每个阶段都需要状态或日志支撑。单个方法上的 `@Transactional` 不能包住 MySQL、Redis和Feign调用形成全局事务。


## 17. 验证清单与故障排查

以下是后续搭建课程环境时的验证计划，**本次没有执行这些服务测试**。

| 验证场景 | 预期结果 | 观察点 |
|---|---|---|
| 立即任务 | 数据库待任务及日志生成，任务进入topic | taskId、status、List |
| 未来2分钟任务 | 先进入future，到期后进入topic | ZSet score是否为毫秒 |
| 未来30分钟任务 | 初始仅数据库，进入缓存窗口后加载 | reloadData查询边界 |
| 单组顺序 | 早到期任务优先具备消费资格 | score、搬迁顺序、入队方式 |
| 取消未到期任务 | 日志为CANCELLED，任务不进入有效审核 | 返回值及缓存两端清理 |
| 修改发布时间 | 旧版本任务被拒绝，新任务按新时间处理 | 任务关联ID、文章版本 |
| Redis暂时写入失败 | 数据库记录可供补偿 | 重试、恢复延迟 |
| 多实例刷新 | 锁与搬迁不产生重复有效审核 | 锁token、taskId、幂等结果 |
| poll后消费端异常 | 暴露讲义的丢任务窗口；改进后可重试 | ACK、处理状态、超时恢复 |
| 审核失败/转人工 | 沿用第04天状态2/3，不误标发布 | reason与文章状态 |
| article服务失败 | 可追踪且可补偿，不重复创建文章 | Feign响应、文章ID |

排查建议按链路进行：

1. **没有任务：** 检查是否草稿、登记是否在事务提交后、Feign响应是否成功。
2. **任务不进缓存：** 检查时间戳单位、5分钟窗口、reloadData是否启用、Redis配置。
3. **future不搬迁：** 检查定时注解、SCAN逻辑、刷新锁是否过期、ZSet score。
4. **topic有数据但不消费：** 检查type/priority是否匹配、Feign路径、响应解析和自媒体调度。
5. **任务已出队但文章没发布：** 检查审核状态、反序列化、外部审核响应及article服务日志。
6. **文章重复或提前审核：** 检查重复任务、改期旧任务、业务版本与保存幂等。

上线时可监控到期未执行数量、最老就绪任务等待时间、任务登记失败率、审核耗时、重试次数和失败任务数量，以业务完成情况衡量效果。


## 18. 复习与自测

### 必须记住的主线

- MySQL是待任务的持久化来源；Redis是近期任务的调度缓存。
- ZSet存尚未到期的近期任务，List存到期可消费任务。
- 每分钟refresh负责搬迁，每5分钟reloadData负责加载/恢复，自媒体每秒poll负责领取。
- SCAN减小单次遍历压力；Pipeline减少网络往返；两者都不自动提供业务原子性。
- 分布式锁减少刷新竞争，过期时间、释放token和幂等仍然重要。
- 讲义取消代码应改为 `CANCELLED`，并返回真实结果。
- 领取完成不是审核完成，生产需要ACK、重试、超时恢复与幂等。
- 本日将“直接审核”改为“到发布时间后审核”，不保证指定时刻已经发布。

### 自测题与参考答案

1. **为什么不把所有任务放入Redis？** 远期任务未必近期执行，数据库持久化加有限缓存窗口能控制缓存规模，并提供待任务恢复来源。
2. **任务的executeTime用秒还是毫秒？** DTO和ZSet score均使用毫秒时间戳，和 `System.currentTimeMillis()` 比较；数据库日期转换必须保持一致。
3. **Pipeline能保证ZREM与LPUSH同时成功吗？** 不能，它优化网络往返；Redis内部原子搬迁可用Lua，但仍不覆盖数据库和业务执行。
4. **乐观锁能防止文章重复发布吗？** 不能单独做到；它检查某次数据库版本更新，APP保存和任务执行仍需业务幂等。
5. **为什么reloadData不能找回所有失败任务？** 讲义poll已删除待任务并标为执行，之后的业务失败不在待表查询结果中。
6. **提前支付后，订单关闭任务被取消就绝对安全吗？** 不一定，任务可能已被领取；执行入口还须复核最新支付状态。
7. **文章设为10:00发布，10:00就能看到吗？** 不能承诺，任务到期后还要等待刷新、消费、审核、保存和详情准备。
8. **生产中如何接近准点发布？** 提前完成审核，到点仅改变发布可见性；仍需调度精度、状态复核、幂等与故障补偿。

讲义末尾“作业”只有标题，未展示具体题目。本节验证清单和自测题是笔记补充，不能当作原讲义作业。
